# 30.5 Logistic Regression Implementation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
from sklearn.datasets import make_classification

- This is going to create a dataset, in the classification approach, basically it will create some independent feature and it will create output feature.
- In the output feature we can say how many categories we want.
- And it will create in such a way that we dont even have to do the standarization. 

In [3]:
X,y = make_classification(n_samples=1000, n_features=10, n_classes=2, n_redundant=2, random_state=54)

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [5]:
# Model tRaining
from sklearn.linear_model import LogisticRegression
logi = LogisticRegression()

In [6]:
logi.fit(X_train, y_train)

LogisticRegression()

In [7]:
y_pred = logi.predict(X_test)

In [8]:
logi.predict_proba(X_test)

array([[9.94242561e-01, 5.75743886e-03],
       [2.79647812e-03, 9.97203522e-01],
       [1.44809699e-02, 9.85519030e-01],
       [9.96276607e-01, 3.72339349e-03],
       [9.29441366e-03, 9.90705586e-01],
       [2.28339115e-02, 9.77166089e-01],
       [3.43602884e-04, 9.99656397e-01],
       [2.76456557e-02, 9.72354344e-01],
       [9.99941383e-01, 5.86173127e-05],
       [5.45559430e-03, 9.94544406e-01],
       [1.00434749e-02, 9.89956525e-01],
       [9.99579528e-01, 4.20472177e-04],
       [3.89031188e-03, 9.96109688e-01],
       [9.39726087e-01, 6.02739133e-02],
       [9.97578926e-01, 2.42107444e-03],
       [9.85584204e-04, 9.99014416e-01],
       [1.41149329e-02, 9.85885067e-01],
       [7.99765498e-01, 2.00234502e-01],
       [5.25675849e-02, 9.47432415e-01],
       [3.19205467e-02, 9.68079453e-01],
       [2.09674832e-02, 9.79032517e-01],
       [3.35727985e-02, 9.66427202e-01],
       [9.98552180e-01, 1.44782041e-03],
       [9.98819283e-01, 1.18071719e-03],
       [9.990959

We use probability to find on which class the new data point will lie...So in this, which ever class has the highest probability it will be the output.

In [9]:
# Performance Martix
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

score = accuracy_score(y_test, y_pred)
print("Accuracy Score: ",score)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix: \n",cm)

print("Classification Report: \n",classification_report(y_test, y_pred))

Accuracy Score:  0.9833333333333333
Confusion Matrix: 
 [[143   4]
 [  1 152]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.99      0.97      0.98       147
           1       0.97      0.99      0.98       153

    accuracy                           0.98       300
   macro avg       0.98      0.98      0.98       300
weighted avg       0.98      0.98      0.98       300



# 30.6 Hyperparameter Tuning and Cross Validation

## Logistic Regression Parameters

- Penalty
  - none
  - l2
  - l1
  - elasticnet
  (these three are ridge, lasso and elasticnet)
- C
- Solver

etc.


In [10]:
model = LogisticRegression()


In [11]:
penalty = ['l1', 'l2']
solver = ['liblinear', 'saga'] 
c_values = [100, 10, 1.0, 0.1, 0.01]
params = dict(penalty=penalty, solver=solver, C=c_values)

Here we will play with all this key-value pairs and findout which one will be the best parameter for our model.

### M1) Grid Search CV

In [12]:
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold()

In [13]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(estimator=model, param_grid=params, scoring='accuracy', cv=cv, n_jobs=-1)

In [14]:
grid

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'C': [100, 10, 1.0, 0.1, 0.01],
                         'penalty': ['l1', 'l2'],
                         'solver': ['liblinear', 'saga']},
             scoring='accuracy')

In [15]:
grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'C': [100, 10, 1.0, 0.1, 0.01],
                         'penalty': ['l1', 'l2'],
                         'solver': ['liblinear', 'saga']},
             scoring='accuracy')

In [16]:
grid.best_params_

{'C': 1.0, 'penalty': 'l1', 'solver': 'liblinear'}

In [17]:
grid.best_score_

0.9942857142857143

In [18]:
y_pred = grid.predict(X_test)

In [19]:
# Performance Martix
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

score = accuracy_score(y_test, y_pred)
print("Accuracy Score: ",score)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix: \n",cm)

print("Classification Report: \n",classification_report(y_test, y_pred))

Accuracy Score:  0.9833333333333333
Confusion Matrix: 
 [[143   4]
 [  1 152]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.99      0.97      0.98       147
           1       0.97      0.99      0.98       153

    accuracy                           0.98       300
   macro avg       0.98      0.98      0.98       300
weighted avg       0.98      0.98      0.98       300



### 2) Randomized SearchCV

- In the previous method we have seen Grid Search CV, in which it usually takes all the combination of all the parameters that are given.
- Due to this combinations, it takes more amount of time.
- To overcome this we use Randomized Search CV. It will make sure to pickup random attributes or random parameters, and then it will try to create some, or it will try to find out what parameter will be suitable for a specific problem statement.

In [20]:
from sklearn.model_selection import RandomizedSearchCV

In [21]:
model = LogisticRegression()
randomcv = RandomizedSearchCV(estimator=model, param_distributions=params, cv=5, scoring='accuracy')

In [22]:
randomcv.fit(X_train, y_train)

c:\Users\OM MAKWANA\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\OM MAKWANA\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


RandomizedSearchCV(cv=5, estimator=LogisticRegression(),
                   param_distributions={'C': [100, 10, 1.0, 0.1, 0.01],
                                        'penalty': ['l1', 'l2'],
                                        'solver': ['liblinear', 'saga']},
                   scoring='accuracy')

In [23]:
randomcv.best_score_

0.9928571428571429

In [24]:
randomcv.best_params_

{'solver': 'liblinear', 'penalty': 'l1', 'C': 100}

In [25]:
y_pred = randomcv.predict(X_test)